In [0]:
from pyspark.sql.functions import max as spark_max
import requests
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp


In [0]:

max_year = (
    spark.table("bronze.model_years")
         .filter("model_year <> 9999")
         .select(spark_max("model_year").alias("max_year"))
         .collect()[0]["max_year"]
)

print(max_year)

In [0]:
# 1. parameters
model_year = max_year  # or the max_year value from before
url = "https://api.nhtsa.gov/products/vehicle/makes"
params = {
    "modelYear": model_year,
    "issueType": "c"
}

# 2. call API
resp = requests.get(url, params=params)
resp.raise_for_status()  # raise exception if status is not 200

data = resp.json()  # parse response as JSON

# 3. parse results
rows = [
    Row(make = m["make"])
    for m in data.get("results", [])
    if m.get("make") is not None  # filter only results with non-null 'make'
]


In [0]:
# 4. create spark df
# Create a Spark DataFrame from the parsed API results and add metadata columns
df_makes = spark.createDataFrame(rows) \
    .withColumn("model_year", lit(model_year)) \
    .withColumn("ingestion_ts", current_timestamp()) \
    .withColumn("source_url", lit(url + "?modelYear=" + str(model_year) + "&issueType=c"))

In [0]:
from delta.tables import DeltaTable

target_table = "bronze.vehicle_makes"

# Remove duplicates based on make and model_year
df_makes = df_makes.dropDuplicates(["make", "model_year"])

# Check if the table exists
table_exists = spark.catalog.tableExists(target_table)

if table_exists:
    dt = DeltaTable.forName(spark, target_table)

    # Merge new data into the existing Delta table
    (dt.alias("t")
      .merge(df_makes.alias("s"),
             "t.make = s.make AND t.model_year = s.model_year")
      .whenMatchedUpdate(set={
          "ingestion_ts": "current_timestamp()",
          "source_url": "s.source_url"
      })
      .whenNotMatchedInsert(values={
          "make": "s.make",
          "model_year": "s.model_year",
          "ingestion_ts": "current_timestamp()",
          "source_url": "s.source_url"
      })
      .execute()
    )
else:
    # Create the Delta table if it does not exist
    (df_makes.write.format("delta")
      .mode("overwrite")
      .saveAsTable(target_table)
    )